In [2]:
import os
import json
import csv


In [3]:

# 定义目录和文件路径
dataset = 'HARDVS' # 'HARDVS', 'PAF', 'DVS128Gesture', 'SeAct'
root_dir = '/root/autodl-tmp/{}'.format(dataset)
base_dir = '/root/autodl-tmp/{}_Sampled_EZCLIP'.format(dataset)
# label_mapping_file = '/root/wj/EZ_CLIP/AFE/{}/{}_idx_to_label.json'.format(dataset) # just for SeACT
dataset_train_file = '/root/wj/EZ_CLIP/AFE/{}/train_label.txt'.format(dataset, dataset)
dataset_val_file = '/root/wj/EZ_CLIP/AFE/{}/val_label.txt'.format(dataset,dataset)
train_output_file = '/root/wj/EZ_CLIP/dataset_splits/{}/Zero-shot/train.txt'.format(dataset)
val_output_file = '/root/wj/EZ_CLIP/dataset_splits/{}/Zero-shot/val.txt'.format(dataset)
idx_mapping_file = '/root/wj/EZ_CLIP/AFE/{}/{}.json'.format(dataset,dataset)
output_csv_file_labels = '/root/wj/EZ_CLIP/lists/{}_labels.csv'.format(dataset)
# description_file = '/root/wj/EZ_CLIP/AFE/{}/{}_ds_cls.json'.format(dataset) # just for SeAct
output_csv_file_description = '/root/wj/EZ_CLIP/GPT_discription/{}_gpt_Class_discription_new.csv'.format(dataset)
output_file_paths = '/root/wj/EZ_CLIP/AFE/{}/{}.txt'.format(dataset,dataset)


# HARDVS.txt

In [4]:

# 获取所有叶结点文件的绝对路径
def get_leaf_files(dir_path):
    leaf_files = []
    action_range = ['action_' + '{:03d}'.format(i+1) for i in range(47)]
    for root, dirs, files in os.walk(dir_path):
        for file in files:
            # 检查文件路径是否在 action_range 内
            if any(action in root for action in action_range):
                leaf_files.append(os.path.join(root, file))
    return leaf_files

# 将文件路径写入文本文件
def write_paths_to_file(file_paths, output_file):
    with open(output_file, 'w') as f:
        for path in file_paths:
            f.write(f"{path}\n")

# 获取所有叶结点文件的路径并写入文本文件
leaf_files = get_leaf_files(root_dir)
write_paths_to_file(leaf_files, output_file_paths)
print(f"文件路径已写入 {output_file_paths}")

文件路径已写入 /root/wj/EZ_CLIP/AFE/HARDVS/HARDVS.txt


# train.txt & val.txt

In [4]:
import os

# 定义你想要加在路径前面的根路径
base_path = '/root/autodl-tmp/HARDVS_Sampled_EZCLIP'

# 定义一个函数来处理文件
def process_labels_file(input_file, output_file):
    # 打开原始文件读取内容
    with open(input_file, 'r') as f:
        lines = f.readlines()

    # 打开输出文件准备写入
    with open(output_file, 'w') as f_out:
        for line in lines:
            # 拆分每行，得到路径、图片数量和标签
            parts = line.strip().split()
            relative_path = parts[0]  # 相对路径
            label = parts[2]  # 标签

            # 组合绝对路径
            full_path = os.path.join(base_path, relative_path)

            # 计算该路径下的 jpg 文件数量
            if os.path.exists(full_path):
                jpg_files = [file for file in os.listdir(full_path) if file.endswith('.jpg')]
                jpg_count = len(jpg_files)
                if jpg_count <= 0:
                    print(f'number <= 0: {full_path}')
                    continue
            else:
                print(f"路径不存在: {full_path}")
                continue
            

            # 生成新的行并写入输出文件
            new_line = f"{full_path} {jpg_count} {label}\n"
            f_out.write(new_line)

    print(f"{output_file} 文件已成功生成！")

# 处理 train_labels.txt
process_labels_file(dataset_train_file, train_output_file)

# 处理 val_labels.txt
process_labels_file(dataset_val_file, val_output_file)


路径不存在: /root/autodl-tmp/HARDVS_Sampled_EZCLIP/action_006/dvSave-2021_07_30_10_44_40
路径不存在: /root/autodl-tmp/HARDVS_Sampled_EZCLIP/action_006/dvSave-2021_07_30_10_45_19
路径不存在: /root/autodl-tmp/HARDVS_Sampled_EZCLIP/action_006/dvSave-2021_07_30_10_45_34
路径不存在: /root/autodl-tmp/HARDVS_Sampled_EZCLIP/action_006/dvSave-2021_07_30_10_45_54
路径不存在: /root/autodl-tmp/HARDVS_Sampled_EZCLIP/action_006/dvSave-2021_07_30_10_46_07
路径不存在: /root/autodl-tmp/HARDVS_Sampled_EZCLIP/action_006/dvSave-2021_07_30_10_46_20
路径不存在: /root/autodl-tmp/HARDVS_Sampled_EZCLIP/action_006/dvSave-2021_07_30_10_46_36
/root/wj/EZ_CLIP/dataset_splits/HARDVS/Zero-shot/train.txt 文件已成功生成！
/root/wj/EZ_CLIP/dataset_splits/HARDVS/Zero-shot/val.txt 文件已成功生成！


# labels.csv

In [7]:
import json,csv
# 生成label的CSV文
with open(idx_mapping_file, 'r') as f:
    idx_mapping = json.load(f)

# 创建一个反向映射以便快速查找类名
idx_to_class_name = [[value-1,key] for key, value in idx_mapping.items()]


# 写入CSV文件
with open(output_csv_file_labels, 'w', newline='') as csvfile:
    csvwriter = csv.writer(csvfile)
    csvwriter.writerow(['id', 'name'])
    for row in idx_to_class_name:
        csvwriter.writerow(row)

print(f"CSV file has been created at {output_csv_file_labels}")

CSV file has been created at /root/wj/EZ_CLIP/lists/HARDVS_labels.csv


# description.txt

In [ ]:
# 生成label的CSV文件
with open(idx_mapping_file, 'r') as f:
    idx_mapping = json.load(f)
ctx_prompt = 'A series of photos recording human action for '
# 创建一个反向映射以便快速查找类名
idx_classname_des = [[value,key,'\n\n' + ctx_prompt + key] for key, value in idx_mapping.items()]
# 写入CSV文件
with open(output_csv_file_description, 'w', newline='') as csvfile:
    csvwriter = csv.writer(csvfile)
    csvwriter.writerow(['SNo', 'Class Name', 'GPT3 discription'])
    for row in idx_classname_des:
        csvwriter.writerow(row)

print(f"CSV file has been created at {output_csv_file_description}")


In [8]:
import pandas as pd

# 输入和输出文件路径
input_csv_file = output_csv_file_description
output_csv_file = output_csv_file_description

# 读取原始 CSV 文件
df = pd.read_csv(input_csv_file)

# 调整 'SNo' 列，使其从 0 开始
df['SNo'] = df.index

# 为 'GPT3 discription' 列的每一行添加两行空白的文本
df['GPT3 discription'] = df['GPT3 discription'].apply(lambda x: f'\n\n{x}')

# 将修改后的 DataFrame 写入新的 CSV 文件
df.to_csv(output_csv_file, index=False)